# Approval node: human-in-the-loop with `interrupt()`

Each scenario starts a fresh thread from `sample_state.json` (research already done),
so only the writer and critic call the LLM. `InMemorySaver` stands in for Postgres.

In [ ]:
import json

from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

from content_agent.graph.builder import build_graph
from content_agent.graph.nodes.critic import _word_count

load_dotenv()
graph = build_graph(checkpointer=InMemorySaver())   # interrupt() needs a checkpointer

with open("research/sample_state.json") as f:
    saved = json.load(f)

In [ ]:
def start_at_writer(thread_id: str) -> dict:
    """New thread whose state is the saved research, as if research_source had just run."""
    config = {"configurable": {"thread_id": thread_id}}
    graph.update_state(config, saved, as_node="research_source")
    print("next:", graph.get_state(config).next)   # ('writer',)
    return config


def show_pause(result: dict) -> None:
    """Print what the reviewer would see."""
    request = result["__interrupt__"][0].value
    print(f"verdict: {request['critic_verdict']}   revisions: {request['revision_count']}   "
          f"words: {_word_count(request['draft'])}")
    if request["error"]:
        print("ERROR:", request["error"])
    if request["critic_note"]:
        print("critic note:\n" + request["critic_note"])
    print("\n" + request["draft"] + "\n")
    for n, url in enumerate(request["references"], 1):
        print(f"[{n}] {url}")

## 1. Approve

`invoke(None, config)` means "continue this thread from its checkpoint". The run pauses at
`approval` and the payload comes back under `__interrupt__`.

In [ ]:
config = start_at_writer("approve-1")
result = graph.invoke(None, config)
show_pause(result)
print("\npaused at:", graph.get_state(config).next)

In [ ]:
result = graph.invoke(Command(resume={"action": "approve", "feedback": None}), config)
print("status:", result["status"])
print("next:", graph.get_state(config).next)   # () = finished

## 2. Request changes

The editor asks for ~120 words. Checks: the writer follows the editor over its "about 200 words"
prompt, `revision_count` restarts at 0, and the critic does not fail the draft on length.

In [ ]:
config = start_at_writer("changes-1")
result = graph.invoke(None, config)
print("first draft words:", _word_count(result["__interrupt__"][0].value["draft"]))

result = graph.invoke(
    Command(resume={"action": "request_changes",
                    "feedback": "Cut it to about 120 words. Keep only the fine amounts and who they apply to."}),
    config,
)
show_pause(result)

In [ ]:
state = graph.get_state(config).values
print("human_feedback:", state["human_feedback"])
print("critic notes this run:", len(state["critic_notes"]))
print("last critic note:", state["critic_notes"][-1])

result = graph.invoke(Command(resume={"action": "approve", "feedback": None}), config)
print("status:", result["status"])

## 3. Invalid answer, then reject

A `request_changes` with blank feedback is re-asked (a second `interrupt()` in the same node):
the run stays paused with an `error` in the payload. No LLM call is made for it.

In [ ]:
config = start_at_writer("reject-1")
result = graph.invoke(None, config)

result = graph.invoke(Command(resume={"action": "request_changes", "feedback": "   "}), config)
print("error:", result["__interrupt__"][0].value["error"])
print("still paused at:", graph.get_state(config).next)

In [ ]:
result = graph.invoke(
    Command(resume={"action": "reject", "feedback": "Topic too narrow for the newsletter."}),
    config,
)
print("status:", result["status"])
print("human_feedback:", result["human_feedback"])
print("next:", graph.get_state(config).next)